# Lung cancer survival prediction

Ordered workflow: load the cleaned data, prepare features, split, preprocess, train, evaluate, and inspect feature importance.

## 1. Imports

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, average_precision_score, balanced_accuracy_score, classification_report, confusion_matrix, f1_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
RANDOM_STATE = 42

## 2. Load the cleaned dataset

In [2]:
data_path = Path('../data/lung_cancer_dataset_cleaned.csv')
if not data_path.exists():
    data_path = Path('data/lung_cancer_dataset_cleaned.csv')
data = pd.read_csv(data_path)
print(f'Dataset shape: {data.shape}')
display(data.head())

Dataset shape: (2000, 31)


,Diagnosis_Year,Diagnosis_Date,WHO_Region,Country,Age,Gender,Cigarettes_Per_Day,Years_Smoking,Secondhand_Smoke,Family_History,...,Weight_Loss,Wheezing,Recurrent_Infections,Swallowing_Difficulty,Finger_Clubbing,Cancer_Type,NSCLC_Subtype,Cancer_Stage,Treatment,Survived
0,2020,2020-11-04,Western Pacific,China,46,Female,28,28,1,0,...,1,0,0,1,0,NSCLC,Adenocarcinoma,Stage IV,Chemo + Radiation,0
1,2021,2021-12-09,Americas,Mexico,71,Male,5,34,1,0,...,0,1,0,1,0,NSCLC,Adenocarcinoma,Stage I,Surgery + Chemotherapy,1
2,2017,2017-04-05,Western Pacific,Japan,53,Female,0,0,1,0,...,0,0,0,0,0,NSCLC,Squamous Cell,Stage I,Surgery,0
3,2018,2018-11-24,Western Pacific,Singapore,55,Male,21,34,0,1,...,1,1,1,0,0,NSCLC,Adenocarcinoma,Stage IV,Chemotherapy,0
4,2024,2024-02-19,Western Pacific,Malaysia,54,Male,0,0,0,0,...,0,1,0,1,0,NSCLC,Adenocarcinoma,Stage III,Immunotherapy,1


## 3. Separate target and features

In [3]:
y = pd.to_numeric(data['Survived'], errors='raise').astype(int)
X = data.drop(columns='Survived').copy()
print('Target counts:')
display(y.value_counts().rename(index={0: 'Not survived', 1: 'Survived'}))
print(f'Input columns: {X.shape[1]}')

Target counts:


Survived
Not survived    1244
Survived         756
Name: count, dtype: int64

Input columns: 30


## 4. Convert the date into model features

The original date string is replaced by year and month. No columns removed in the cleaning notebook are recreated.

In [4]:
if 'Diagnosis_Date' in X.columns:
    diagnosis_date = pd.to_datetime(X.pop('Diagnosis_Date'), errors='coerce')
    X['Diagnosis_Date_Year'] = diagnosis_date.dt.year
    X['Diagnosis_Date_Month'] = diagnosis_date.dt.month
print(f'Features after date preparation: {X.shape[1]}')

Features after date preparation: 31


## 5. Split into training and test data

In [5]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE)
print(f'Train rows: {len(X_train)} | Test rows: {len(X_test)}')

Train rows: 1600 | Test rows: 400


## 6. Define preprocessing

Numeric columns are median-imputed and scaled. Text columns are filled with the most common value and one-hot encoded.

In [6]:
numeric_features = X_train.select_dtypes(include=np.number).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=np.number).columns.tolist()
numeric_pipeline = Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())])
categorical_pipeline = Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('one_hot', OneHotEncoder(handle_unknown='ignore'))])
preprocessor = ColumnTransformer([('numeric', numeric_pipeline, numeric_features), ('categorical', categorical_pipeline, categorical_features)])
print('Numeric features:', numeric_features)
print('Categorical features:', categorical_features)

Numeric features: ['Diagnosis_Year', 'Age', 'Cigarettes_Per_Day', 'Years_Smoking', 'Secondhand_Smoke', 'Family_History', 'Occupational_Hazard', 'BMI', 'Chronic_Lung_Disease', 'Asbestos_Exposure', 'Radon_Exposure', 'Previous_Cancer_History', 'Coughing', 'Shortness_of_Breath', 'Chest_Pain', 'Coughing_Blood', 'Weight_Loss', 'Wheezing', 'Recurrent_Infections', 'Swallowing_Difficulty', 'Finger_Clubbing', 'Diagnosis_Date_Year', 'Diagnosis_Date_Month']
Categorical features: ['WHO_Region', 'Country', 'Gender', 'Genetic_Mutation', 'Cancer_Type', 'NSCLC_Subtype', 'Cancer_Stage', 'Treatment']


## 7. Define and train models

In [7]:
models = {
    'Dummy baseline': DummyClassifier(strategy='most_frequent'),
    'Logistic Regression': LogisticRegression(class_weight='balanced', max_iter=2000, random_state=RANDOM_STATE),
    'Random Forest': RandomForestClassifier(n_estimators=300, class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=150, random_state=RANDOM_STATE)
}
pipelines = {}
for name, model in models.items():
    pipelines[name] = Pipeline([('preprocessing', preprocessor), ('model', model)])
    pipelines[name].fit(X_train, y_train)
    print(f'{name} trained successfully.')

Dummy baseline trained successfully.


Logistic Regression trained successfully.


Random Forest trained successfully.


Gradient Boosting trained successfully.


## 8. Evaluate on the held-out test set

In [8]:
def evaluate_model(name, pipeline):
    prediction = pipeline.predict(X_test)
    probability = pipeline.predict_proba(X_test)[:, 1]
    return {'Model': name, 'Accuracy': accuracy_score(y_test, prediction), 'Balanced accuracy': balanced_accuracy_score(y_test, prediction), 'ROC-AUC': roc_auc_score(y_test, probability), 'PR-AUC': average_precision_score(y_test, probability), 'F1 (not survived)': f1_score(y_test, prediction, pos_label=0), 'F1 (survived)': f1_score(y_test, prediction, pos_label=1)}
test_results = pd.DataFrame([evaluate_model(name, pipe) for name, pipe in pipelines.items()])
display(test_results.set_index('Model').round(4).sort_values('ROC-AUC', ascending=False))

,Accuracy,Balanced accuracy,ROC-AUC,PR-AUC,F1 (not survived),F1 (survived)
Model,,,,,,
Gradient Boosting,0.6950,0.6873,0.7722,0.6114,0.7458,0.6188
Logistic Regression,0.6950,0.7081,0.7628,0.6003,0.7277,0.6534
Random Forest,0.7000,0.6900,0.7590,0.6038,0.7521,0.6203
Dummy baseline,0.6225,0.5000,0.5000,0.3775,0.7673,0.0000


## 9. Inspect the best model

In [9]:
best_name = test_results.sort_values('ROC-AUC', ascending=False).iloc[0]['Model']
best_pipeline = pipelines[best_name]
best_prediction = best_pipeline.predict(X_test)
print(f'Best model by ROC-AUC: {best_name}')
display(pd.DataFrame(confusion_matrix(y_test, best_prediction), index=['Actual not survived', 'Actual survived'], columns=['Predicted not survived', 'Predicted survived']))
print(classification_report(y_test, best_prediction, target_names=['Not survived', 'Survived']))

Best model by ROC-AUC: Gradient Boosting


,Predicted not survived,Predicted survived
Actual not survived,179,70
Actual survived,52,99


              precision    recall  f1-score   support

Not survived       0.77      0.72      0.75       249
    Survived       0.59      0.66      0.62       151

    accuracy                           0.69       400
   macro avg       0.68      0.69      0.68       400
weighted avg       0.70      0.69      0.70       400



## 10. Compare models with five-fold cross-validation

In [10]:
scoring = {'accuracy': 'accuracy', 'balanced_accuracy': 'balanced_accuracy', 'roc_auc': 'roc_auc', 'f1_macro': 'f1_macro'}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_rows = []
for name, pipe in pipelines.items():
    scores = cross_validate(pipe, X, y, cv=cv, scoring=scoring, n_jobs=-1)
    cv_rows.append({'Model': name, **{f'{metric} mean': scores[f'test_{metric}'].mean() for metric in scoring}, **{f'{metric} std': scores[f'test_{metric}'].std() for metric in scoring}})
cv_results = pd.DataFrame(cv_rows).set_index('Model')
display(cv_results.round(4).sort_values('roc_auc mean', ascending=False))

,accuracy mean,balanced_accuracy mean,roc_auc mean,f1_macro mean,accuracy std,balanced_accuracy std,roc_auc std,f1_macro std
Model,,,,,,,,
Random Forest,0.7230,0.7049,0.7887,0.7048,0.0135,0.0128,0.0161,0.0127
Gradient Boosting,0.7175,0.7020,0.7817,0.7008,0.0158,0.0154,0.0193,0.0153
Logistic Regression,0.7275,0.7428,0.7808,0.7235,0.0277,0.0294,0.0198,0.0280
Dummy baseline,0.6220,0.5000,0.5000,0.3835,0.0010,0.0000,0.0000,0.0004


## 11. Inspect Random Forest feature importance

This is for interpretation only; it does not modify the cleaned data.

In [11]:
forest_pipeline = pipelines['Random Forest']
encoded_names = forest_pipeline.named_steps['preprocessing'].get_feature_names_out()
forest = forest_pipeline.named_steps['model']
importance = pd.Series(forest.feature_importances_, index=encoded_names).sort_values(ascending=False)
display(importance.head(20).to_frame('importance'))

,importance
categorical__Cancer_Stage_Stage IV,0.085390
categorical__Cancer_Stage_Stage I,0.069582
numeric__BMI,0.053350
numeric__Age,0.049706
categorical__Cancer_Stage_Stage III,0.041829
numeric__Diagnosis_Date_Month,0.037446
numeric__Diagnosis_Date_Year,0.033827
numeric__Diagnosis_Year,0.033115
categorical__Cancer_Stage_Stage II,0.030762
categorical__Treatment_Surgery,0.029320


## 12. Fit and save the best model for the GUI

The best model is selected by cross-validated ROC-AUC, refit using all cleaned data, and saved together with the feature layout and explanation metadata.

In [ ]:
import joblib
from datetime import datetime

# Select the strongest model by mean cross-validated ROC-AUC.
best_cv_name = cv_results['roc_auc mean'].idxmax()
best_cv_pipeline = pipelines[best_cv_name]

# Refit the selected pipeline on all available cleaned data.
best_cv_pipeline.fit(X, y)

# Store the exact columns expected by the GUI.
model_bundle = {
    'model': best_cv_pipeline,
    'model_name': best_cv_name,
    'input_columns': X.columns.tolist(),
    'numeric_features': numeric_features,
    'categorical_features': categorical_features,
    'cv_results': cv_results.reset_index(),
    'saved_at': datetime.now().isoformat(timespec='seconds')
}

model_output_path = Path('../data/best_survival_model.joblib')
if not model_output_path.parent.exists():
    model_output_path = Path('data/best_survival_model.joblib')
joblib.dump(model_bundle, model_output_path)
print(f'Saved model: {best_cv_name}')
print(f'Saved to: {model_output_path}')